# Chapter 11.1. 신뢰 영역과 확률비 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter11_1_trust_region_ratio.ipynb)

책 본문: [11.1 신뢰 영역과 확률비](https://smhanlab.com/book-ml/kor/ml2/chapter11/1.html)

책 11.1절의 두 가지를 실제로 실행해 봅니다.

1. **확률비 \(r_t = \pi_	heta / \pi_{\text{old}}\)**를 2-행동 softmax와 가우시안 정책에서 직접 계산해,
   \(r_t\)가 1에서 출발해 정책이 움직이는 만큼 벗어나는 것을 확인합니다.
2. **중요도 가중치의 분산이 \(e^{\mu^2}-1\)로 지수적으로 불어나는 것**을
   이산(정규식)과 연속(가우시안 시뮬레이션) 두 가지로 검증하고,
   "소수 극단 샘플이 업데이트를 지배한다"는 현상을 눈으로 확인합니다.

결론: \(r_t\)가 1에서 멀어질수록 중요도 샘플링의 한 샘플이
전체 업데이트를 지배하게 되고, 이것이 PPO 클리핑(11.2절)이 존재하는
이유입니다.

## 1. 설정: 한국어 폰트와 저장 위치

마크다운 셀의 \( \) / \[ \] 수식은 Colab에서 렌더링됩니다.
그림의 한국어 라벨은 'Noto Sans CJK KR' 폰트를 씁니다(없으면 기본 폰트).
저장 위치는 로컬에서는 저장소 images 폴더, Colab에서는 /tmp로 자동 대체됩니다.

In [1]:
import os, math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager

kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr:
    plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print(f"그림 저장 위치: {IMG}")

그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 2. 확률비 \(r_t\): 2-행동 softmax (책 11.1절 "손으로 한번" 표 재현)

\(\theta_{\text{old}}=(0,0)\)에서 두 행동을 반반(\(p_0=0.5\))으로
고르던 정책이, 행동 0이 좋다고 배우며 \(\theta_0\)만 올리면
\(r_0 = p_0/0.5\)가 어떻게 변하는지 직접 계산합니다.

In [2]:
def softmax2(logits):
    e = np.exp(np.array(logits, float) - max(logits))
    return e / e.sum()

p_old = softmax2([0.0, 0.0])
print(f"시작점 theta=(0,0): p0_old = {p_old[0]:.4f},  r0 = 1.000 (새·옛 정책이 같음)")
print()
print(f"{'theta':>14s}  {'p0 (새)':>9s}  {'r0 = p0/0.5':>12s}")
for t0 in [0.0, 1.1, 2.2, 4.4]:
    p = softmax2([t0, 0.0])
    r0 = p[0] / p_old[0]
    print(f"({t0:>4.1f}, 0.0)  {p[0]:9.4f}  {r0:12.3f}")
print()
print("-> r0는 1에서 출발해 1보다 커지며, 2-행동 이산 정책에서는 p0<=1이므로 r0<=2.")
print("   즉 '위에서 유한'하지만 경계(2)에 빠르게 붙는다. 클리핑이 막는 것이 바로 이 구간.")

시작점 theta=(0,0): p0_old = 0.5000,  r0 = 1.000 (새·옛 정책이 같음)

         theta     p0 (새)   r0 = p0/0.5
( 0.0, 0.0)     0.5000         1.000
( 1.1, 0.0)     0.7503         1.501
( 2.2, 0.0)     0.9002         1.800
( 4.4, 0.0)     0.9879         1.976

-> r0는 1에서 출발해 1보다 커지며, 2-행동 이산 정책에서는 p0<=1이므로 r0<=2.
   즉 '위에서 유한'하지만 경계(2)에 빠르게 붙는다. 클리핑이 막는 것이 바로 이 구간.


## 3. 확률비가 '첫 스텝에서' policy gradient와 같음을 직접 확인

\(\nabla_\theta L^{IS} = \mathbb{E}[r_t \, \nabla_\theta \log\pi_\theta\, A_t]\) 이고,
\(\theta=\theta_{\text{old}}\)에서 \(r_t=1\)이므로 policy gradient와 같아야 합니다.
2-행동 softmax에서 \(\theta=(0,0)\)에서, 행동 0을 고른 샘플(\(A_0=+1\)) 하나에 대해
두 그래디언트를 손으로(미분 공식으로) 계산해 비교합니다.

one-vs-rest 공식(10.2절): \(\nabla_\theta \log\pi(0|s) = [(1-p_0),\, -p_1]\cdot s\), 여기 \(s=1\).

In [3]:
import torch

# 2-행동 softmax, s=1.  theta_old=(0,0) -> pi_old(행동0|s)=0.5.
theta = torch.tensor([0.0, 0.0], requires_grad=True)
logits = theta * 1.0            # s = 1
p = torch.softmax(logits, dim=0)
pi_old = 0.5                    # 옛 정책의 행동0 확률 (상수)
A = torch.tensor(1.0)           # 이 샘플의 어드밴티지 (스칼라, theta와 무관)

# L_IS = r * A,  r = pi/pi_old.  (A는 상수이므로 미분에는 안 들어감)
r = p[0] / pi_old
L_IS = r * A
L_IS.backward()
grad_IS = theta.grad.clone()

# policy gradient (10.3): A * grad log pi(a|s)  -> one-vs-rest, s=1, a=0
grad_pg = torch.tensor([(1 - 0.5), -0.5]) * 1.0

print("L_IS 그래디언트    :", [round(x,4) for x in grad_IS.tolist()])
print("policy grad (A*g)  :", [round(x,4) for x in grad_pg.tolist()])
print("일치:", torch.allclose(grad_IS, grad_pg, atol=1e-6))
print()
print("-> theta=theta_old에서 r=1이므로 L_IS의 그래디언트가 policy gradient와 정확히 같음.")
print("   (r이 1에서 멀어지면 이 동등성이 약해진다 -> 11.2절 클리핑의 역할)")

L_IS 그래디언트    : [0.5, -0.5]
policy grad (A*g)  : [0.5, -0.5]
일치: True

-> theta=theta_old에서 r=1이므로 L_IS의 그래디언트가 policy gradient와 정확히 같음.
   (r이 1에서 멀어지면 이 동등성이 약해진다 -> 11.2절 클리핑의 역할)


## 4. 중요도 가중치의 분산: 이산(정규식) vs 가우시안(시뮬레이션)

가우시안 정책 \(\pi_{\text{old}}=\mathcal{N}(0,1)\),
\(\pi_\theta=\mathcal{N}(\mu,1)\)에서
\(r=\exp(\mu x-\mu^2/2)\)\(\ (x\sim\mathcal{N}(0,1))\)는
\(\mathbb{E}[r]=1\), \(\operatorname{Var}[r]=e^{\mu^2}-1\)입니다.
\(\mu\)를 0.5~3.0까지 바꿔가며, **정규식**과 **10만 샘플 시뮬레이션**
두 가지로 분산을 계산해 비교합니다. 주목할 점: \(\mu\)가 클수록
시뮬레이션 값이 정규식과 **큰 편차**를 보이는데, 이건 실험의 실패가 아니라
"tail이 무거워서 **분산 자체를** 유한 샘플로 추정하기 어렵다"는 현상입니다.

In [4]:
def ratio_var_analytic(mu):
    return math.exp(mu*mu) - 1.0

mus = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
print(f"{'mu':>4s}  {'Var(e^mu^2-1)':>14s}  {'시뮬레이션(n=1e5)':>18s}  {'E[w]':>7s}  {'max(w)':>10s}")
for mu in mus:
    va = ratio_var_analytic(mu)
    rng = np.random.default_rng(0); x = rng.normal(size=100_000)
    w = np.exp(mu*x - mu*mu/2)
    print(f"{mu:4.1f}  {va:14.3f}  {w.var():18.3f}  {w.mean():7.3f}  {w.max():10.1f}")
print()
print("-> mu<=1.5: 정규식과 시뮬레이션이 일치 (E[w]=1 확인).")
print("   mu>=2: 시뮬레이션 분산이 정규식에서 크게 벗어남 -- max(w)~1744의 한 샘플의")
print("   제곱(~3e6)이 전체 제곱합과 같은 크기라, '분산의 추정'조차 tail에 지배당함.")
print("   -> 'tail이 지배한다'는 말은 평균뿐 아니라 분산 자체에도 적용된다.")

  mu   Var(e^mu^2-1)        시뮬레이션(n=1e5)     E[w]      max(w)
 0.5           0.284               0.285    1.000         9.4
 1.0           1.718               1.767    1.000        68.9
 1.5           8.488               9.571    1.003       392.7
 2.0          53.598              66.592    1.018      1744.0
 2.5         517.013             501.944    1.052      6032.4
 3.0        8102.084            3025.756    1.115     16250.4

-> mu<=1.5: 정규식과 시뮬레이션이 일치 (E[w]=1 확인).
   mu>=2: 시뮬레이션 분산이 정규식에서 크게 벗어남 -- max(w)~1744의 한 샘플의
   제곱(~3e6)이 전체 제곱합과 같은 크기라, '분산의 추정'조차 tail에 지배당함.
   -> 'tail이 지배한다'는 말은 평균뿐 아니라 분산 자체에도 적용된다.


## 5. "한 샘플이 지배한다": mu=2에서 실제 10,000개 샘플

책 본문 표의 \(\mu=2\) 행을, **실제 샘플**로 만들어 봅니다.
가중치의 평균은 1 근처지만, 최댓값과 상위 1%가 차지하는
'가중치 질(mass)'을 측정해, 업데이트가 소수 극단 샘플에 의해
정해짐을 확인합니다. (시드 고정, 재현 가능)

In [5]:
rng = np.random.default_rng(0)
x = rng.normal(size=10_000)
mu = 2.0
w = np.exp(mu*x - mu*mu/2)

sw = w.sum()
top100 = np.sort(w)[::-1][:100]   # 상위 1% (10,000개 중 100개)
print(f"mu={mu}, n=10,000")
print(f"  가중치 평균       = {w.mean():.4f}  (중요도 샘플링 -> ~1)")
print(f"  가중치 최댓값     = {w.max():.2f}")
print(f"  상위 1%(100개)가  = {top100.sum()/sw:.1%}  (전체 가중치 합 대비)")
print(f"  w>1인 샘플 비율   = {np.mean(w>1):.1%}")
print()
print("-> 평균은 1인데 최댓값 ~143, 상위 1%가 전체의 ~31%를 차지.")
print("   업데이트 방향을 정하는 것은 '대부분의 샘플'이 아닌 소수 극단 샘플.")

mu=2.0, n=10,000
  가중치 평균       = 0.9096  (중요도 샘플링 -> ~1)
  가중치 최댓값     = 143.12
  상위 1%(100개)가  = 30.6%  (전체 가중치 합 대비)
  w>1인 샘플 비율   = 16.1%

-> 평균은 1인데 최댓값 ~143, 상위 1%가 전체의 ~31%를 차지.
   업데이트 방향을 정하는 것은 '대부분의 샘플'이 아닌 소수 극단 샘플.


## 6. 그림: 분산의 지수적 불어나기 + 한 샘플 지배 (책 본문 그림)

왼쪽: \(\operatorname{Var}[r]=e^{\mu^2}-1\)이 \(\mu^2\)에 지수적으로
불어나는 곡선(로그 축). 오른쪽: \(\mu=2\)에서 상위 1%가 전체 가중치
합의 ~31%를 차지함을 막대로 시각화. 이 그림이 책 11.1절에
\(ch11\_1\_ratio\_variance.svg\)로 저장됩니다.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# --- 왼쪽: Var(r) = e^{mu^2} - 1 ---
mus_fine = np.linspace(0.1, 3.0, 300)
var_fine = np.exp(mus_fine**2) - 1.0
ax = axes[0]
ax.semilogy(mus_fine, var_fine, color="tab:blue", lw=2)
for mu, col in [(0.5,"tab:gray"),(1.0,"tab:green"),(1.5,"tab:orange"),(2.0,"tab:red"),(2.5,"tab:purple"),(3.0,"tab:brown")]:
    v = math.exp(mu*mu)-1.0
    ax.scatter([mu],[v], color=col, zorder=5, s=34)
    ax.annotate(f"mu={mu}: {v:.2f}", (mu,v), textcoords="offset points",
                xytext=(6,-10), fontsize=8, color=col)
ax.set_xlabel("평균 이동 mu (새-옛 정책)")
ax.set_ylabel("Var[r] = e^{mu^2} - 1  (로그)")
ax.set_title("중요도 가중치 분산: 지수적 불어나기")
ax.grid(alpha=0.3, which="both")

# --- 오른쪽: mu=2, 상위 1%가 질을 지배 ---
rng = np.random.default_rng(0)
x = rng.normal(size=10_000)
w = np.exp(2.0*x - 2.0)
order = np.argsort(w)[::-1]
frac_mass = np.cumsum(w[order]) / w.sum()
ax = axes[1]
top_frac = np.array([1,5,10,25,50,100,250,500])/100.0
top_mass = [frac_mass[int(f*10000)-1] for f in [0.01,0.05,0.10,0.25,0.50,1.0]]
labels = ["상위 1%","5%","10%","25%","50%","100%"]
bars = ax.bar(range(len(labels)), top_mass, color="tab:red", alpha=0.85)
for i,(lab,v) in enumerate(zip(labels,top_mass)):
    ax.text(i, v+0.02, f"{v:.0%}", ha="center", fontsize=9)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
ax.set_ylabel("상위 k%가 차지하는 가중치 질")
ax.set_ylim(0,1.15)
ax.set_title("mu=2: 소수 극단 샘플이 질을 지배")
ax.grid(alpha=0.3, axis="y")

fig.tight_layout()
fig.savefig(IMG + "/ch11_1_ratio_variance.svg", bbox_inches="tight")
plt.show()
print("저장:", IMG + "/ch11_1_ratio_variance.svg")

저장: /home/smhan/book-ml/kor/src/images/ch11_1_ratio_variance.svg


## 7. 종합: 확률비가 '재사용의 열쇠'인 이유와 그 대가

1. \(L^{IS}=\mathbb{E}[r_t A_t]\)의 그래디언트는
   \(\mathbb{E}[r_t \nabla_\theta\log\pi_\theta\, A_t]\) —
   policy gradient에 \(r_t\)가 곱해진 형태. \(\theta=\theta_{\text{old}}\)에서
   \(r_t=1\)이라 **첫 스텝은 정직한 policy gradient**. (3절에서 직접 검증)
2. 그러나 \(r_t\)가 1에서 멀어지면, 가중치 분산이 \(e^{\mu^2}-1\)로
   지수적으로 불어나 **소수 극단 샘플이 업데이트를 지배**. (4~6절)
3. 이 두 사실(첫 걸음은 정직, 큰 걸음은 위험)이 **11.2절의 클리핑**을
   필요로 한다 — \(r_t\)가 \([1-\epsilon,\,1+\epsilon]\)를 벗어나면
   기여를 0으로 만들어, "한 샘플이 지배할 수 있는 최대 크기"를 제한한다.